# Traveling Purchaser Problem — Genetic Algorithm

The **Traveling Purchaser Problem (TPP)** extends the TSP: a purchaser must buy `k` items across `n` markets (cities). Every city has a price for every item. The goal is to find a **full Hamiltonian tour** (all cities visited, like TSP) and decide **where to buy each item** along that tour, minimising total **travel cost + purchasing cost**.

### Chromosome representation
A solution is a **permutation of all `n` cities** — the visit order. Purchasing is resolved greedily from the tour: scan cities in order; at each city buy any item whose cheapest-so-far price is beaten. A city may be passed through without buying anything.

Crossover uses the inversion-vector technique from the TSP notebook; mutation swaps two positions.

In [ ]:
using Evolutionary
using Random
using Plots

# ── Problem generator ─────────────────────────────────────────────────────────
function generate_random_tpp(n, k; random_seed = 1)
    rng = Random.MersenneTwister(random_seed)
    X = 100 * rand(rng, n)
    Y = 100 * rand(rng, n)
    d = [sqrt((X[i] - X[j])^2 + (Y[i] - Y[j])^2) for i in 1:n, j in 1:n]
    cost_item_city = 50 .+ 100 .* rand(rng, k, n)   # cost_item_city[item, city]
    return X, Y, d, cost_item_city
end

n = 100   # number of cities / markets
k = 100   # number of items to purchase
X, Y, d, cost_item_city = generate_random_tpp(n, k)
println("Generated TPP: n=$n cities, k=$k items")

In [ ]:
# ── Inversion-vector encoding (same as TSP notebook) ─────────────────────────

function perm_to_inv(perm::Vector)
    N = length(perm)
    inv = zeros(Int, N)
    for i in 1:N
        m = 1
        while perm[m] != i
            if perm[m] > i
                inv[i] += 1
            end
            m += 1
        end
    end
    return inv
end

function inv_to_perm(inv::Vector)
    N = length(inv)
    pos = zeros(Int, N)
    for i in N:-1:1
        for m in (i+1):N
            if pos[m] >= inv[i] + 1
                pos[m] += 1
            end
        end
        pos[i] = inv[i] + 1
    end
    perm = zeros(Int, N)
    for i in 1:N
        perm[pos[i]] = i
    end
    return perm
end

# Sanity check
RP = randperm(20)
println("Inversion encoding roundtrip OK: ", RP == inv_to_perm(perm_to_inv(RP)))

In [ ]:
# ── TPP decoder: permutation → purchasing assignment ─────────────────────────
#
# ALL cities are visited in the order given by the permutation (full tour, like TSP).
# Purchasing strategy: greedy along the tour.
#   Scan cities in tour order. At each city, buy every item for which this city
#   offers the lowest price among the cities not yet passed.
#   Equivalently: each item is bought at the cheapest city encountered first —
#   i.e. argmin over cost_item_city[item, perm[1:end]] broken by position.
#
# Travel cost: full closed tour (last city → first city included, like TSP).

function decode_tpp(perm::Vector{Int}, d::Matrix{Float64}, cost_item_city::Matrix{Float64})
    n = size(d, 1)
    k = size(cost_item_city, 1)

    # Travel cost: closed Hamiltonian tour over all cities
    travel_cost = sum(d[perm[i], perm[mod1(i+1, n)]] for i in 1:n)

    # For each item, find the city in the tour that offers the lowest price.
    # Ties broken by position in tour (buy earlier).
    purchase_cost = 0.0
    items_bought_at = zeros(Int, k)   # for diagnostics
    for item in 1:k
        best_price = Inf
        best_city  = perm[1]
        for city in perm
            if cost_item_city[item, city] < best_price
                best_price = cost_item_city[item, city]
                best_city  = city
            end
        end
        purchase_cost += best_price
        items_bought_at[item] = best_city
    end

    return perm, travel_cost, purchase_cost, items_bought_at
end

# Quick test
test_perm = collect(1:n)
_, tc0, pc0, _ = decode_tpp(test_perm, d, cost_item_city)
println("Test decode: all $n cities visited, travel=$(round(tc0,digits=2)), purchase=$(round(pc0,digits=2)), total=$(round(tc0+pc0,digits=2))")

In [ ]:
# ── Genetic operators ─────────────────────────────────────────────────────────

"""Swap-mutation on a permutation (no subtour penalty needed for TPP)."""
function mutate_swap_tpp(recombinant::AbstractVector;
        rng::AbstractRNG = Random.default_rng())
    N = length(recombinant)
    i = rand(rng, 1:N)
    j = rand(rng, 1:N)
    recombinant[i], recombinant[j] = recombinant[j], recombinant[i]
    return recombinant
end

"""Inversion-vector crossover (same as TSP notebook, valid for TPP)."""
function recombine_tpp(v1::T, v2::T;
        rng::AbstractRNG = Random.default_rng()) where {T <: AbstractVector}
    N = length(v1)
    i1 = perm_to_inv(v1)
    i2 = perm_to_inv(v2)
    cp = rand(rng, 2:(N-1))
    c1 = vcat(i1[1:cp], i2[(cp+1):N])
    c2 = vcat(i2[1:cp], i1[(cp+1):N])
    return inv_to_perm(c1), inv_to_perm(c2)
end

In [ ]:
# ── Fitness function ──────────────────────────────────────────────────────────

function tpp_cost(perm::Vector{Int})
    _, travel, purchase, _ = decode_tpp(perm, d, cost_item_city)
    return travel + purchase
end

x0 = collect(1:n)
println("Initial (identity permutation) cost: ", round(tpp_cost(x0), digits=2))

In [ ]:
# ── Run Genetic Algorithm ─────────────────────────────────────────────────────

x_sol = Evolutionary.optimize(
    tpp_cost,
    x0,
    GA(;
        populationSize = 500,
        crossover      = recombine_tpp,
        mutation       = mutate_swap_tpp,
        epsilon        = 0.01,
        crossoverRate  = 0.9,
        mutationRate   = 0.05,
    ),
    Evolutionary.Options(; iterations = 500, show_trace = true),
)

println("\nBest total cost: ", round(x_sol.minimum, digits=2))
best_perm = x_sol.minimizer
_, best_travel, best_purchase, _ = decode_tpp(best_perm, d, cost_item_city)
println("  Travel cost   : ", round(best_travel,   digits=2))
println("  Purchase cost : ", round(best_purchase, digits=2))

In [ ]:
# ── Visualise the solution ────────────────────────────────────────────────────

function plot_tpp_solution(X, Y, perm, items_bought_at)
    n = length(perm)
    # Count items bought at each city
    items_per_city = zeros(Int, n)
    for city in items_bought_at
        items_per_city[city] += 1
    end

    plt = Plots.plot(title="TPP solution — all $n cities, coloured by items bought", legend=false)

    # Draw tour edges
    for i in 1:n
        a = perm[i]
        b = perm[mod1(i+1, n)]
        Plots.plot!(plt, [X[a], X[b]], [Y[a], Y[b]]; color=:lightblue, lw=1)
    end

    # Draw cities: grey if nothing bought, blue shaded by volume
    for city in 1:n
        cnt = items_per_city[city]
        col = cnt == 0 ? :lightgrey : :steelblue
        sz  = cnt == 0 ? 4 : 5 + cnt
        scatter!(plt, [X[city]], [Y[city]]; color=col, markersize=sz, alpha=0.8)
    end

    # Mark start
    scatter!(plt, [X[perm[1]]], [Y[perm[1]]]; color=:green, markersize=10, markershape=:star5)
    return plt
end

_, best_travel, best_purchase, items_bought_at = decode_tpp(best_perm, d, cost_item_city)
plot_tpp_solution(X, Y, best_perm, items_bought_at)

## Experiment: population size & iterations

Run the GA with several configurations and collect results.

In [ ]:
configs = [
    (pop=100,  iters=200),
    (pop=500,  iters=200),
    (pop=500,  iters=500),
    (pop=1000, iters=500),
]

using Printf
println("pop_size | iterations | best_cost  | travel    | purchase  | time(s)")
println(repeat("-", 68))

for cfg in configs
    t0 = time()
    sol = Evolutionary.optimize(
        tpp_cost, x0,
        GA(;
            populationSize = cfg.pop,
            crossover      = recombine_tpp,
            mutation       = mutate_swap_tpp,
            epsilon        = 0.01,
            crossoverRate  = 0.9,
            mutationRate   = 0.05,
        ),
        Evolutionary.Options(; iterations = cfg.iters),
    )
    elapsed = round(time() - t0, digits=1)
    _, tv, pv, _ = decode_tpp(sol.minimizer, d, cost_item_city)
    @printf("%8d | %10d | %10.2f | %9.2f | %9.2f | %7.1f\n",
        cfg.pop, cfg.iters, sol.minimum, tv, pv, elapsed)
end